## Example 1: jupyter_client functionality test 1

In [ ]:
import jupyter_client
import os

# Example 1: Working with KernelManager to manage kernel lifecycle
print("Example 1: KernelManager - Starting and managing a Python kernel")

km = jupyter_client.KernelManager(kernel_name='python3')
try:
    # Start the kernel
    km.start_kernel()
    print(f"Kernel started: {km.is_alive()}")
    print(f"Kernel ID: {km.kernel_id}")
    print(f"Connection file: {os.path.basename(km.connection_file)}")

    # Get kernel info
    kc = km.client()
    kc.start_channels()
    kc.kernel_info()
    reply = kc.get_shell_msg(timeout=2)
    
    # Safely access kernel info
    if isinstance(reply, dict) and 'content' in reply:
        lang_info = reply['content'].get('language_info', {})
        lang_name = lang_info.get('name', 'unknown') if isinstance(lang_info, dict) else 'unknown'
        print(f"Kernel language: {lang_name}")
finally:
    # Clean up
    km.shutdown_kernel()
    print("Kernel shut down successfully")

## Example 2: jupyter_client functionality test 2

In [ ]:
import jupyter_client
import os

# Example 2: Discovering available kernels using KernelSpecManager
print("Example 2: KernelSpecManager - Listing available kernels")

# Create a KernelSpecManager instance
ksm = jupyter_client.kernelspec.KernelSpecManager()

# Get all available kernel specs
kernels = ksm.get_all_specs()
print(f"Found {len(kernels)} kernel(s)")

for kernel_name, kernel_spec in kernels.items():
    print(f"\nKernel: {kernel_name}")
    # kernel_spec is a dict with 'spec' and 'resource_dir' keys
    spec = kernel_spec['spec']
    print(f"  Display name: {spec['display_name']}")
    print(f"  Language: {spec['language']}")
    print(f"  Resource dir: {os.path.basename(kernel_spec['resource_dir'])}")

# Get the default kernel
default_kernel = ksm.kernel_name if hasattr(ksm, 'kernel_name') else 'python3'
print(f"\nDefault kernel: {default_kernel}")

# Check version
print(f"\njupyter_client version: {jupyter_client.__version__}")

## Example 3: jupyter_client functionality test 3

In [ ]:
import jupyter_client
from jupyter_client import BlockingKernelClient

# Example 3: Execute code using BlockingKernelClient
print("Example 3: BlockingKernelClient - Executing code synchronously")

km = jupyter_client.KernelManager(kernel_name='python3')
km.start_kernel()

try:
    # Create a blocking client using the kernel manager's client
    kc = km.client()
    kc.start_channels()

    # Execute simple code
    msg_id = kc.execute("result = 2 + 3\nprint(f'Result: {result}')")

    # Get the execution reply
    reply = kc.get_shell_msg(timeout=5)
    print(f"Execution status: {reply['content']['status']}")

    # Get output from iopub channel
    output_received = False
    timeout_count = 0
    while not output_received and timeout_count < 5:
        try:
            msg = kc.get_iopub_msg(timeout=1)
            if msg['msg_type'] == 'stream':
                print(f"Output: {msg['content']['text'].strip()}")
                output_received = True
        except:
            timeout_count += 1
finally:
    kc.stop_channels()
    km.shutdown_kernel()

## Example 4: jupyter_client functionality test 4

In [ ]:
import jupyter_client
from jupyter_client import MultiKernelManager
import asyncio

# Example 4: Managing multiple kernels with MultiKernelManager
print("Example 4: MultiKernelManager - Managing multiple kernels")

# Create MultiKernelManager
mkm = MultiKernelManager()

# Start multiple kernels
kernel_ids = []
for i in range(2):
    kernel_id = mkm.start_kernel(kernel_name='python3')
    kernel_ids.append(kernel_id)
    print(f"Started kernel {i+1}: {kernel_id[:8]}...")

# List all running kernels
print(f"\nActive kernels: {len(mkm.list_kernel_ids())}")

# Get info about each kernel
for kid in kernel_ids:
    km = mkm.get_kernel(kid)
    print(f"Kernel {kid[:8]}... is alive: {km.is_alive()}")

# Shutdown all kernels
for kid in kernel_ids:
    mkm.shutdown_kernel(kid)
print("\nAll kernels shut down")

## Example 5: jupyter_client functionality test 5

In [ ]:
import jupyter_client
from jupyter_client.connect import find_connection_file
import json

# Example 5: Working with connection files and kernel provisioning
print("Example 5: Connection files and kernel provisioning")

# Start a kernel and work with its connection file
km = jupyter_client.KernelManager(kernel_name='python3')
km.start_kernel()

try:
    # Get connection info
    conn_file = km.connection_file
    print(f"Connection file: {conn_file}")
    
    # Read and display connection parameters
    with open(conn_file, 'r') as f:
        conn_info = json.load(f)
    
    print(f"Transport: {conn_info.get('transport', 'N/A')}")
    print(f"IP: {conn_info.get('ip', 'N/A')}")
    print(f"Kernel name: {conn_info.get('kernel_name', 'N/A')}")
    
    # Check if provisioner is available (jupyter_client 7.0+)
    if hasattr(km, 'provisioner'):
        print(f"Provisioner: {km.provisioner.__class__.__name__ if km.provisioner else 'None'}")
finally:
    km.shutdown_kernel()
    print("Kernel shut down")